# Part 7 — Evaluation code, shared pixel mask, trivial baseline

**Goal:**
1. the unit tests of the shared evaluation code (`src/eval`) pass;
2. `processed/valid_mask.npy`: the pixels used for training and evaluation by every variant of both methods (not void, data in every embedding);
3. a trivial baseline (most frequent training class everywhere) on official fold I, logged in `results/runs.csv`: the lower bound of every table and an end-to-end check of the evaluation (per-patch confusion matrices, boundary/core, bootstrap).

Evaluation protocol: mIoU over the 19 classes 0–18 (background included, void ignored), plus OA, macro-F1, IoU per class, mIoU on parcel boundaries (≤ 1 px from a change of parcel id) and cores. A class absent from both ground truth and predictions is left out of the means. The crop-only variant used by the RF pilot (18 classes, background pixels left out) is also printed.

**Needs:** Parts 1, 3 and 5 on the Drive. **Runtime:** CPU. **Time:** about 5 min.

## 1. Runtime, Drive and configuration

In [ ]:
import sys, shutil, platform
from pathlib import Path
import json, numpy as np, pandas as pd
print("Python", platform.python_version())

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"
CODE = Path("/content/code")
PROC_DRIVE = DRIVE / "processed"
RUNS = DRIVE / "runs"
RUNS_CSV = DRIVE / "results" / "runs.csv"

## 2. Code

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt

## 3. Unit tests of the evaluation code
Small examples computed by hand: confusion matrix (void and invalid pixels ignored), IoU and F1, absent classes, crop-only scores, boundary/core masks, bootstrap and paired difference, `runs.csv` writing.

**Check:** the last line says `10 passed` (or more) and nothing failed.

In [ ]:
!python -m pytest -q tests

## 4. Shared pixel mask
**Check:** the script prints the void pixels, the pixels removed by each embedding (TESSERA 2019 about 1,000, TESSERA 2018 about 7,000, AlphaEarth 0) and saves `valid_mask.npy`.

In [ ]:
!python scripts/build_valid_mask.py --processed "{PROC_DRIVE}"

## 5. Trivial baseline on fold I
Predicts the most frequent class of the training pixels (folds 1-2-3) everywhere on the test fold (5).

**Check:** one line of scores is printed and `results/runs.csv` holds the run `base_majority_100_foldI_s0`. Expected: a very low mIoU (one class right, all others at 0), OA equal to the share of that class among the test pixels, boundary and core mIoU at 0 if the majority class is the background (background pixels belong to neither region).

In [ ]:
!python scripts/baseline_majority.py --processed "{PROC_DRIVE}" --split I --runs-dir "{RUNS}" --runs-csv "{RUNS_CSV}"

In [ ]:
runs = pd.read_csv(RUNS_CSV)
pd.set_option("display.width", 200)
print(runs[["run_id", "miou", "oa", "macro_f1", "miou_boundary", "miou_core", "n_test_patches", "commit"]].to_string(index=False))
print(sorted(p.name for p in (RUNS / "base_majority_100_foldI_s0").iterdir()))